In [1]:
# Imports necesarios
import torch
import torch.nn as nn

In [2]:
import os
os.chdir("/home/carlo/cuarto/GPT2_MOE/src")

In [3]:
%load_ext autoreload
%autoreload 2

In [6]:
# Relative imports
from gpt2_moe.train_test.utils import Utils
from gpt2_moe.model import GPTMOE

In [46]:
def train(training_params:dict, model:GPTMOE, attr:dict, name:str,  mode:bool, version:str, id:int = None)->None:
    """
        Function aimed to conduct training

        Params:
            training_params(dict):
                Params related to training
            model(GPTMOE):
                The model we are dealing with

            attr(dict):
                The model attributes
            name(str):
                The name of the model
            version(str):
                The training version we are dealing with
        
            mode(bool):
                Enables expert|FF or router training

            id(int):
                The current expert|FF we are training


        
    
    """
    # Base path
    base_path = 'gpt2_moe/checkpoints'

    # Initialize model 
    model = GPTMOE(attr)
    blocks = attr['n_layers']

    # Get params
    epochs = training_params.get('epochs', 2)
    bs = training_params.get('bs', 4)
    data_train = [1,2,3]
    data_val = [1,2,3]


    # 1. LOADING
    # 1.1 Load the base model 
    Utils.load_weights(model = model, name = name, base_path = base_path, blocks = blocks)
    # 1.2 Load the other components
    if not mode:
        Utils.load_all(model = model, base_path = base_path, version = version)

    # 2.FREEZING
    Utils.freeze_selective(model = model, train_mode = mode)

    # 3. TRAINING
    # Define structures
    train_loss = []
    val_loss = []
    # Reference values
    best_val_loss = float('inf')
   


    for i in range(epochs):
        # Current vals
        val_loss = 0
        train_loss = 0
        # Training data
        for batch in data_train:
            # Feed
            # Loss
            # Update
            pass
        # Validation data
        for batch  in data_val:
            # Feed
            # Loss
            pass

        # 4. STORING 
        if best_val_loss > val_loss:
            # Store params
            Utils.params_storage(model = model, base_path = base_path, version = version, train_mode = mode, id = id)
            




Hay un bug que no tengo tiempo de solucionar ahora, el código está pensado para entrenar los expertos de uno en uno, pero el problema es que hay bloques en que puede haber ffn y yo lo tengo puesto para que lo guarde y congele y descongele a los expertos y ffn de forma indistinta. Hay que ver cómo se resuelve eso. Por el resto,  en principio bien. 

He corregido una cosa de forma muy guarra en la funcion de almacenamiento porque cuando llamas varias veces por validacion, cuando usaba get next name, como que me almacenaba tantas veces como entraba a dicha bloque if en el train, es decir, que a lo mejor en la ejecucion de train donde entreno al expert 0, me guardaba en su bloque correspondiente expertos del 0 al K, cuando realmente son el mismo pero en distintos momentos del ciclo de train. Por tanto, le paso un id guarro para conformar el nombre, mira a ver si lo puedes arreglar.

In [47]:
# Define params
params = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers":2,  
    "emb_dim": 768, 
    "dropout":0.1,
    "moe_mask":
        {
            "schedule": "all",
            "offset":2

        },

    "ffn_hidden_dim": None,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":1,
        "k_routing":1,
        "hidden_dim":768,
    },
        
    }


In [48]:
for i in range(3):
    # Define params
    train_params = {"epochs":10, "bs":2}
    # Set model and name
    model = GPTMOE
    name = "gpt2-small-124M.pth"
    # Define the mode : expert training -> uno a uno 
    mode = True
    id = i

    train(training_params = train_params,
        model = model, attr = params, name = name, 
        mode = mode, version = 'version_0',
        id = id)


(general_load) LOADING BASE MODEL WEIGHTS

(freezing)FREEZING STAGE
·Heating experts and ffn

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(general_load) LOADING BASE MODEL WEIGHTS

(freezing)FREEZING STAGE
·Heating experts and ffn

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) STORING PARAMS
·Conducting (experts|ff) storing

(params_storage) ST

In [49]:
 # Define params
train_params = {"epochs":10, "bs":2}
# Set model and name
model = GPTMOE
name = "gpt2-small-124M.pth"
# Define the mode : expert training -> uno a uno 
mode = False
id = i

train(training_params = train_params,
    model = model, attr = params, name = name, 
    mode = mode, version = 'version_0',
    id = id)


(general_load) LOADING BASE MODEL WEIGHTS

(load_all)LOADING AVAILABLE BLOCKS
·(blocks_loader)Loading block (0)
 ->(expert_load) Loading expert (0)
     ·Aspect of the full path: gpt2_moe/checkpoints/version_0/0/expert_0.pth
·(blocks_loader)Loading block (1)
 ->(expert_load) Loading expert (0)
     ·Aspect of the full path: gpt2_moe/checkpoints/version_0/1/expert_0.pth

(freezing)FREEZING STAGE
·Heating the router

(params_storage) STORING PARAMS
·Conducting (whole model) storing
(next_name) Aspect of the new idx: 0
(next_name) Aspect of the new name: model_0.pth

(params_storage) STORING PARAMS
·Conducting (whole model) storing
(next_name) Aspect of the last used idx: 0
(next_name) Aspect of the new idx: 1
(next_name) Aspect of the new name: model_1.pth

(params_storage) STORING PARAMS
·Conducting (whole model) storing
(next_name) Aspect of the last used idx: 1
(next_name) Aspect of the new idx: 2
(next_name) Aspect of the new name: model_2.pth

(params_storage) STORING PARAMS
·Condu